# Laboratorio 12b — Una mini-GAN que aprende a dibujar dígitos
## Versión **estudiante**

**Curso:** Machine Learning (EIN143A25) · Paralelo 701
**Fecha:** Miércoles 11 de noviembre de 2026 · Clase 12 · Lab. Central 114
**Bloque:** Lab B · ~12:20–13:00 (40 min), después de la teoría GAN
**Viene de:** teoría GAN (deck «Clase 12 - RNN y GAN», diapos 17–23: falsificador contra policía, entrenamiento por turnos, colapso de modos y el árbol «¿qué red uso?»; la fórmula está en el Anexo B, diapo 25).
**Antes:** Lab 12a (pronóstico con RNN, LSTM y GRU, otro notebook).

| Parte | Tema | Tiempo sugerido |
|---|---|---|
| **B1** | Del ruido a los dígitos | 20 min |
| **B2** | ¿Salen todos los dígitos? | 10 min |
| **B3** | Reflexión | 5 min |
| — | Margen | 5 min |
| **Entregable** | Tabla de tres escenarios | al final |
| *Extra* | *El juego en una dimensión (opcional, no se evalúa)* | *si te sobra tiempo* |

## Objetivos de aprendizaje

1. Ver con tus propios ojos cómo un falsificador que parte dibujando ruido aprende, de a poco, a dibujar dígitos.
2. Entrenar por turnos a las dos redes, el falsificador y el policía, y entender que el falsificador **nunca ve un dígito real**: aprende solo por las pistas del policía.
3. Revisar si salen todos los dígitos o si aparece el **colapso de modos**: el falsificador se queda dibujando unos pocos dígitos.

> Todo el notebook es **offline**: los dígitos vienen incluidos en scikit-learn. Funciona en Colab o en tu computador (basta la CPU).

> **Cómo se trabaja.** Completa las celdas marcadas con `# TODO` (reemplaza los `...`). Justo después de cada una hay un **autochequeo**: una celda que te avisa con un mensaje si algo no cuadra. Si pasa, verás un ✔. Las preguntas se responden en la celda «**Respuesta:**» que sigue a cada una.

In [ ]:
# --- Configuración -----------------------------------------------------------
import sys, time, random
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    %pip install -q numpy pandas matplotlib scikit-learn torch

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import animation
from matplotlib.patches import FancyBboxPatch
from IPython.display import HTML, display
from sklearn.datasets import load_digits
try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("No se encontró PyTorch. En tu computador: activa el entorno virtual e instala con "
                      "`pip install -r requirements.txt`. Si no, abre este notebook en Google Colab.") from e

SEED = 42
RAPIDO = False   # True = menos pasos, solo para ensayar el código (los valores del libreto usan False)
torch.set_num_threads(1)
T0 = time.time()


def fijar_semillas(semilla=SEED):
    """Deja reproducibles random, NumPy y PyTorch."""
    random.seed(semilla)
    np.random.seed(semilla)
    torch.manual_seed(semilla)


fijar_semillas()

# Paleta del deck «Clase 12 - RNN y GAN»
AZUL, NARANJA, MARINO, GRIS, GRIS_CLARO = "#2e69b2", "#ba6826", "#1b2740", "#8b929c", "#d9dbdf"
plt.rcParams.update({
    "figure.facecolor": "#f6f6f4", "savefig.facecolor": "#f6f6f4", "axes.facecolor": "#fffefb",
    "axes.edgecolor": "#dcdad3", "axes.labelcolor": MARINO, "text.color": MARINO,
    "xtick.color": "#5d6675", "ytick.color": "#5d6675", "axes.grid": True, "grid.color": "#e3e1da",
    "grid.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlecolor": MARINO, "figure.dpi": 100, "font.size": 10.5,
    "axes.prop_cycle": plt.cycler(color=[AZUL, NARANJA, MARINO, GRIS]),
})
print("Python", sys.version.split()[0], "| PyTorch", torch.__version__, "| RAPIDO =", RAPIDO, "| Colab =", IN_COLAB)

---
## El juego: falsificador contra policía

> **En palabras simples (diapos 18 y 19).** Dos redes juegan por turnos. El falsificador (en los libros, el *generador*) recibe números al azar y dibuja un dígito falso. El policía (el *discriminador*) mira un dígito y dice qué tan seguro está de que es real. En su turno, el policía practica distinguir; en el suyo, el falsificador ajusta su receta para engañarlo. Ojo: el falsificador **nunca ve un dígito real**. Aprende solo por las pistas que le da el policía.

Es el mismo dibujo de la diapo 18:

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.3))
ax.set(xlim=(0, 10), ylim=(-0.45, 3.0))
ax.axis("off")


def caja(x, y, titulo, detalle, fondo, texto="white", borde="none"):
    ax.add_patch(FancyBboxPatch((x, y), 1.9, 0.9, boxstyle="round,pad=0.04,rounding_size=0.15", fc=fondo, ec=borde, lw=1.8))
    ax.text(x + 0.95, y + 0.58, titulo, ha="center", va="center", color=texto, fontsize=11.5, fontweight="bold")
    ax.text(x + 0.95, y + 0.27, detalle, ha="center", va="center", color=texto, fontsize=9)


def flecha(x0, y0, x1, y1, **kw):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="-|>", color=MARINO, lw=1.8, **kw))


caja(0.1, 1.3, "ruido", "16 números al azar", GRIS)
caja(2.6, 1.3, "falsificador", "dibuja 64 píxeles", NARANJA)
caja(5.1, 1.3, "dígito falso", "lo que dibujó", GRIS_CLARO, texto=MARINO, borde=NARANJA)
caja(5.1, -0.25, "dígito real", "uno de los 1797", GRIS_CLARO, texto=MARINO, borde=MARINO)
caja(7.8, 0.55, "policía", "¿qué tan real? (0 a 1)", AZUL)
flecha(2.0, 1.75, 2.6, 1.75)
flecha(4.5, 1.75, 5.1, 1.75)
flecha(7.0, 1.75, 7.8, 1.25)
flecha(7.0, 0.2, 7.8, 0.85)
ax.annotate("", xy=(3.55, 2.2), xytext=(8.75, 1.45),
            arrowprops=dict(arrowstyle="-|>", color=NARANJA, lw=1.8, ls="--", connectionstyle="arc3,rad=0.25"))
ax.text(6.4, 2.75, "pistas: «así me engañarías mejor»", ha="center", va="center", color=NARANJA, fontsize=10, style="italic")
plt.show()

---
## Parte B1 — Del ruido a los dígitos (20 min)

> **En palabras simples (diapo 19).** Al comienzo el falsificador no sabe nada: sus pesos son números al azar y lo que dibuja es ruido. En cada turno el policía lo pilla, y esa corrección le dice hacia dónde mover sus píxeles para parecer más real. Ronda a ronda, el ruido se va convirtiendo en dígitos.

**Consigna**

1. Mira los dígitos reales y lo que recibe el falsificador.
2. Arma las dos redes (`crear_falsificador` y `crear_policia`).
3. Completa las dos líneas de los **turnos** y entrena **5000 pasos** (lotes de 64).
4. Dale play a la animación y mira cómo el mismo dibujo pasa de ruido a número.

### Los datos

`load_digits()` trae **1797 dígitos escritos a mano**, de 8×8 píxeles. Los escalamos para que cada píxel vaya de 0 (blanco) a 1 (negro).

In [ ]:
digitos_np = load_digits().data / 16.0                  # (1797, 64): cada fila es un dígito de 8×8 = 64 píxeles
digitos = torch.tensor(digitos_np, dtype=torch.float32)
CMAP_DIG = plt.get_cmap("gray_r").copy()                # trazo oscuro sobre fondo claro
CMAP_DIG.set_bad(GRIS_CLARO)                            # color de las líneas que separan los dígitos


def mosaico(imgs, filas, cols):
    """Pega filas × cols dígitos de 8×8 en una sola imagen, separados por una línea gris."""
    imgs = np.asarray(imgs).reshape(filas, cols, 8, 8)
    lienzo = np.full((filas * 9 + 1, cols * 9 + 1), np.nan)
    for r in range(filas):
        for c in range(cols):
            lienzo[1 + 9 * r: 9 + 9 * r, 1 + 9 * c: 9 + 9 * c] = imgs[r, c]
    return lienzo


def mostrar(ax, imagen, titulo=""):
    """Dibuja un dígito (o un mosaico de dígitos) sin ejes."""
    im = ax.imshow(imagen, cmap=CMAP_DIG, vmin=0, vmax=1)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for borde in ax.spines.values():
        borde.set_visible(False)
    ax.set_title(titulo)
    return im


fig, ax = plt.subplots(figsize=(9, 1.7))
mostrar(ax, mosaico(digitos_np[:10], 1, 10), "Esto es lo que el falsificador tiene que aprender a dibujar")
plt.show()
print(f"{len(digitos)} dígitos reales, de {digitos.shape[1]} píxeles cada uno.")

### Lo que recibe el falsificador

Su trabajo es convertir estos 16 números en 64 píxeles que parezcan un dígito. Usaremos siempre **los mismos 16 ruidos** (`Z_FIJO`) para sacarle «fotos» al falsificador durante el entrenamiento: así vemos cómo cambia *el mismo* dibujo.

In [ ]:
DIM_Z = 16                                  # el ruido: 16 números al azar
fijar_semillas(SEED)
Z_FIJO = torch.randn(16, DIM_Z)             # 16 ruidos fijos, los mismos en todas las fotos


def dibujar_ruido(ax, z, titulo=""):
    """Los 16 números al azar como una imagen de 4×4, con el valor escrito en cada casilla."""
    z = np.asarray(z).reshape(4, 4)
    ax.imshow(z, cmap="gray_r", vmin=-2.5, vmax=2.5)
    for (i, j), v in np.ndenumerate(z):
        ax.text(j, i, f"{round(float(v), 1) + 0.0:.1f}".replace(".", ","), ha="center", va="center", fontsize=9.5,
                color="white" if v > 0.5 else MARINO)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.set_title(titulo)


fig, ax = plt.subplots(figsize=(4, 4.3))
dibujar_ruido(ax, Z_FIJO[0], "Esto es lo que recibe el falsificador:\n16 números al azar. No se parece a nada")
plt.show()

### Las dos redes

- **Falsificador:** 16 números al azar → 64 → 64 píxeles. `Linear(16, 64) → ReLU → Linear(64, 64) → Sigmoid`. El Sigmoid del final deja cada píxel entre 0 (blanco) y 1 (negro).
- **Policía:** 64 píxeles → 32 → 1 número. `Linear(64, 32) → LeakyReLU(0.2) → Linear(32, 1) → Sigmoid`. Entrega una probabilidad (de 0 a 1) de que el dígito sea real.

In [ ]:
def crear_falsificador(dim_z=DIM_Z):
    """Falsificador: 16 números al azar -> 64 píxeles entre 0 y 1."""
    # TODO: devuelve nn.Sequential(Linear(dim_z, 64) -> ReLU -> Linear(64, 64) -> Sigmoid)
    ...


def crear_policia():
    """Policía: 64 píxeles -> probabilidad (0 a 1) de que el dígito sea real."""
    # TODO: devuelve nn.Sequential(Linear(64, 32) -> LeakyReLU(0.2) -> Linear(32, 1) -> Sigmoid)
    ...

In [ ]:
# --- Autochequeo: las dos redes ---
_g, _d = crear_falsificador(), crear_policia()
assert _g is not None and _d is not None, "Falta completar crear_falsificador o crear_policia (¿olvidaste el return?)."
_falsos = _g(torch.randn(5, DIM_Z))
assert _falsos.shape == (5, 64), \
    f"El falsificador debería convertir 5 ruidos de 16 números en 5 dibujos de 64 píxeles: forma (5, 64), no {tuple(_falsos.shape)}."
assert 0 <= _falsos.min().item() and _falsos.max().item() <= 1, "Los píxeles deben quedar entre 0 y 1: ¿falta la Sigmoid final del falsificador?"
_p = _d(_falsos)
assert _p.shape == (5, 1), f"El policía debería entregar una probabilidad por dibujo: forma (5, 1), no {tuple(_p.shape)}."
assert 0 <= _p.min().item() and _p.max().item() <= 1, "El policía debe entregar probabilidades entre 0 y 1: ¿falta su Sigmoid final?"
assert sum(p.numel() for p in _g.parameters()) == 16 * 64 + 64 + 64 * 64 + 64, "El falsificador no tiene el tamaño esperado (16 -> 64 -> 64)."
assert sum(p.numel() for p in _d.parameters()) == 64 * 32 + 32 + 32 + 1, "El policía no tiene el tamaño esperado (64 -> 32 -> 1)."
print("✔ Falsificador y policía bien")

### Los turnos

En cada paso del entrenamiento pasan dos cosas, en este orden:

1. **Turno del policía.** Mira un lote de dígitos reales (debería decir 1) y uno de falsos (debería decir 0), y se corrige.
2. **Turno del falsificador.** Dibuja dígitos y se corrige para que el policía diga 1 («real») sobre ellos.

`bce` mide qué tan lejos está la respuesta del policía de lo que debería haber dicho. La receta: Adam con `lr=2e-4` y `betas=(0.5, 0.999)` para las dos redes, lotes de 64 y 5000 pasos.

Las **fotos** se sacan en el paso 0 (antes de entrenar) y luego cada 125 pasos, siempre con `Z_FIJO`. Sacar una foto solo *mira* al falsificador: no lo cambia.

In [ ]:
PASOS_DIG = 1500 if RAPIDO else 5000


def entrenar_gan_digitos(pasos=PASOS_DIG, lote=64, lr=2e-4, foto_cada=125, perdida_cada=50, z_fijo=Z_FIJO, semilla=SEED):
    """Entrena por turnos. Devuelve el falsificador, el policía, las «fotos» de lo que dibuja con `z_fijo`
    (en el paso 0 y cada `foto_cada` pasos) y las pérdidas promedio de cada `perdida_cada` pasos."""
    fijar_semillas(semilla)
    G, D = crear_falsificador(), crear_policia()
    opt_g = torch.optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    opt_d = torch.optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    bce = nn.BCELoss()
    unos, ceros = torch.ones(lote, 1), torch.zeros(lote, 1)

    def foto():
        with torch.no_grad():                  # solo mirar: no corrige nada ni gasta números al azar
            return G(z_fijo).numpy().reshape(-1, 8, 8)

    fotos = {0: foto()}                        # paso 0: antes de entrenar
    perdidas = {"paso": [], "policía": [], "falsificador": []}
    tramo_d, tramo_g = [], []
    for paso in range(1, pasos + 1):
        real = digitos[torch.randint(0, len(digitos), (lote,))]
        falso = G(torch.randn(lote, DIM_Z))

        # Turno del policía: aprende a decir 1 a los dígitos reales y 0 a los falsos.
        # (falso.detach(): en este turno solo se corrige al policía, no al falsificador)
        opt_d.zero_grad()
        # TODO: suma dos errores: bce(lo que dice D sobre `real`, unos) + bce(lo que dice D sobre falso.detach(), ceros)
        loss_d = ...
        loss_d.backward()
        opt_d.step()

        # Turno del falsificador: ajusta su receta para que el policía diga 1 («real») sobre sus dibujos.
        opt_g.zero_grad()
        # TODO: un solo error: bce(lo que dice D sobre `falso`, unos)   (aquí SIN .detach(): ahora se corrige al falsificador)
        loss_g = ...
        loss_g.backward()
        opt_g.step()

        tramo_d.append(loss_d.item())
        tramo_g.append(loss_g.item())
        if paso % perdida_cada == 0:
            perdidas["paso"].append(paso)
            perdidas["policía"].append(np.mean(tramo_d))
            perdidas["falsificador"].append(np.mean(tramo_g))
            tramo_d, tramo_g = [], []
        if paso % foto_cada == 0:
            fotos[paso] = foto()
    return G, D, fotos, perdidas

In [ ]:
# --- Autochequeo: los turnos (un ensayo de 30 pasos) ---
try:
    _, _, _f0, _ = entrenar_gan_digitos(pasos=0)
    _, _, _f, _p = entrenar_gan_digitos(pasos=30, foto_cada=10, perdida_cada=10)
except AttributeError as e:
    raise AssertionError("Falta completar una de las dos líneas de los turnos (todavía hay `...`).") from e
assert sorted(_f) == [0, 10, 20, 30] and _f[0].shape == (16, 8, 8), "Debería quedar una foto en el paso 0 y una cada 10 pasos, de 16 dígitos de 8×8."
assert len(_p["policía"]) == len(_p["falsificador"]) == 3, "Debería quedar una pérdida promedio cada 10 pasos."
assert abs(_p["policía"][0] - 2 * np.log(2)) < 0.35, \
    f"Al comienzo la pérdida del policía debería rondar 1,39 (suma de DOS errores) y es {_p['policía'][0]:.2f}. ¿Sumaste lo real y lo falso?"
assert abs(_p["falsificador"][0] - np.log(2)) < 0.35, \
    f"Al comienzo la pérdida del falsificador debería rondar 0,69 y es {_p['falsificador'][0]:.2f}. ¿Comparaste D(falso) con `unos`?"
assert not np.allclose(_f0[0], _f[30]), "El falsificador no cambió en 30 pasos. En su turno, ¿usaste `falso` SIN .detach()? Así es como se corrige al falsificador."
print("✔ Los turnos están bien armados")

### Entrenar (unos segundos)

In [ ]:
t0 = time.time()
G_dig, D_dig, fotos, perdidas = entrenar_gan_digitos()
pasos_foto = sorted(fotos)
print(f"Entrenamiento listo en {time.time() - t0:.0f} s · {len(fotos)} fotos (paso 0 y cada 125 pasos)")

### Del ruido a los dígitos, en movimiento

Denle play y miren cómo un mismo dibujo pasa de ruido a número. Son los 16 dibujos que salen de los 16 ruidos de `Z_FIJO`, un cuadro cada 125 pasos.

In [ ]:
fig, ax = plt.subplots(figsize=(4.4, 4.8), dpi=80)
imagen = mostrar(ax, mosaico(fotos[0], 4, 4))


def cuadro(k):
    imagen.set_data(mosaico(fotos[pasos_foto[k]], 4, 4))
    ax.set_title(f"paso {pasos_foto[k]} de {PASOS_DIG}")
    return (imagen,)


anim = animation.FuncAnimation(fig, cuadro, frames=len(pasos_foto), interval=250)
plt.close(fig)                            # evita que la figura salga repetida debajo de la animación
HTML(anim.to_jshtml())

### Antes y después, para un mismo ruido

In [ ]:
K = 6                                     # cuál de los 16 ruidos seguimos (puedes cambiarlo: 0 a 15)
fig, ejes = plt.subplots(1, 3, figsize=(10, 3.9))
dibujar_ruido(ejes[0], Z_FIJO[K], "lo que entra\n(16 números al azar)")
mostrar(ejes[1], fotos[0][K], "lo que dibuja en el paso 0\n(antes de entrenar)")
mostrar(ejes[2], fotos[PASOS_DIG][K], f"lo que dibuja en el paso {PASOS_DIG}\n(al final)")
plt.tight_layout()
plt.show()

### Para el entregable: la grilla del entrenamiento

Cada fila es una etapa. Las columnas son los primeros 10 ruidos de `Z_FIJO`: cada columna sigue a *un mismo* dibujo de arriba abajo.

In [ ]:
pasos_grilla = [p for p in (0, 250, 500, 1000, 2000, 3000) if p < PASOS_DIG] + [PASOS_DIG]
filas = [("reales", digitos_np[:10])] + [(f"paso {p}", fotos[p][:10]) for p in pasos_grilla]
fig, ejes = plt.subplots(len(filas), 1, figsize=(9, 0.92 * len(filas) + 0.6))
for ax, (nombre, imgs) in zip(ejes, filas):
    mostrar(ax, mosaico(imgs, 1, 10))
    ax.set_ylabel(nombre, rotation=0, ha="right", va="center", fontsize=10.5)
fig.suptitle("Arriba, dígitos reales. Abajo, lo que dibuja el falsificador con el mismo ruido a lo largo del entrenamiento",
             fontsize=10.5, fontweight="bold", color=MARINO)
plt.tight_layout()
plt.show()

### Las curvas de pérdida

Aquí las curvas **no tienen que bajar**: las dos redes se persiguen. La mejora no se ve en las curvas, se ve en la grilla. Las líneas punteadas marcan el empate ideal (policía ≈ 1,39, falsificador ≈ 0,69); por qué justo esos números está en «Para saber más», al final.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 3.2))
ax.plot(perdidas["paso"], perdidas["policía"], color=MARINO, label="policía")
ax.plot(perdidas["paso"], perdidas["falsificador"], color=NARANJA, label="falsificador")
ax.axhline(2 * np.log(2), color=MARINO, ls=":", lw=1.2)
ax.axhline(np.log(2), color=NARANJA, ls=":", lw=1.2)
ax.set(title="Pérdidas durante el juego (promedio de cada 50 pasos)", xlabel="paso", ylabel="pérdida")
ax.legend(frameon=False)
plt.show()
print(f"Promedio de los últimos 500 pasos -> policía: {np.mean(perdidas['policía'][-10:]):.2f} (empate 1,39) "
      f"| falsificador: {np.mean(perdidas['falsificador'][-10:]):.2f} (empate 0,69)")

In [ ]:
# --- Autochequeo B1: ¿aprendió a dibujar? ---
def variedad(imgs):
    """Cuánto cambian los dibujos entre sí, píxel a píxel (0 = todos iguales)."""
    return np.asarray(imgs).reshape(len(imgs), 64).std(axis=0).mean()


def distancia_al_promedio(imgs):
    """Qué tan lejos está el dibujo promedio del dígito promedio real (error medio por píxel)."""
    return np.abs(np.asarray(imgs).reshape(len(imgs), 64).mean(axis=0) - digitos_np.mean(axis=0)).mean()


assert sorted(fotos) == list(range(0, PASOS_DIG + 1, 125)), "Debería haber una foto en el paso 0 y una cada 125 pasos."
print(f"Variedad entre los 16 dibujos:        paso 0 = {variedad(fotos[0]):.3f} · paso {PASOS_DIG} = {variedad(fotos[PASOS_DIG]):.3f} "
      f"· 16 dígitos reales = {variedad(digitos_np[:16]):.3f}")
print(f"Distancia al dígito promedio real:    paso 0 = {distancia_al_promedio(fotos[0]):.3f} · paso {PASOS_DIG} = {distancia_al_promedio(fotos[PASOS_DIG]):.3f}")
if RAPIDO:
    print("⚠ RAPIDO = True: me salto los chequeos de calidad (con 1500 pasos el falsificador todavía no termina de aprender).")
else:
    MENSAJE = "El falsificador todavía no aprende a dibujar dígitos. Revisa la tasa de aprendizaje (2e-4), los `betas` y el orden de los turnos."
    assert variedad(fotos[PASOS_DIG]) > 0.1, f"{MENSAJE} (Los 16 dibujos finales son casi iguales entre sí.)"
    assert distancia_al_promedio(fotos[PASOS_DIG]) < 0.12, f"{MENSAJE} (Su dibujo promedio no se parece al dígito promedio real.)"
    print("✔ B1 OK: los dibujos finales son variados y, en promedio, se parecen a los dígitos reales")

**Pregunta B1.** ¿En qué paso los dibujos empiezan a parecer dígitos? ¿Qué aparece primero: la forma general o los detalles? Si el falsificador nunca vio un dígito real, ¿cómo aprendió?

**Respuesta:** (escribe aquí)

---
## Parte B2 — ¿Salen todos los dígitos? (10 min)

> **En palabras simples (diapo 20).** Si el falsificador descubre que ciertos dígitos engañan mejor al policía, puede quedarse dibujando solo esos. Eso es el **colapso de modos**: el falsificador se queda dibujando unos pocos dígitos.

**Consigna.** Con el falsificador ya entrenado, generamos **100 dígitos con ruido nuevo** (no `Z_FIJO`) y los ponemos al lado de 100 dígitos reales al azar. Cuenta a ojo qué dígitos aparecen y cuáles faltan, y completa la tabla.

In [ ]:
fijar_semillas(SEED + 1)                    # otra semilla: ruido nuevo, distinto de Z_FIJO
with torch.no_grad():
    inventados = G_dig(torch.randn(100, DIM_Z)).numpy()
reales_100 = digitos_np[np.random.choice(len(digitos_np), 100, replace=False)]

fig, ejes = plt.subplots(1, 2, figsize=(11, 6))
mostrar(ejes[0], mosaico(inventados, 10, 10), "100 dígitos inventados por el falsificador")
mostrar(ejes[1], mosaico(reales_100, 10, 10), "100 dígitos reales al azar")
plt.tight_layout()
plt.show()

**Completa la tabla** (mirando la grilla de la izquierda):

| Dígito | ¿Aparece? | ¿Cuántas veces, aprox.? |
|---|---|---|
| 0 | | |
| 1 | | |
| 2 | | |
| 3 | | |
| 4 | | |
| 5 | | |
| 6 | | |
| 7 | | |
| 8 | | |
| 9 | | |

**Pregunta B2.** ¿Aparecen los 10 dígitos? ¿Hay algunos que se repiten mucho y otros que no salen nunca? ¿Por qué le convendría al falsificador hacer eso?

**Respuesta:** (escribe aquí)

---
## Parte B3 — Reflexión (5 min)

> **En palabras simples (diapos 19 y 20).** Ahora miren hacia atrás. En una GAN lo que importa no es que una curva baje, sino lo que el falsificador dibuja. Estas tres preguntas se responden con lo que vieron en B1 y B2.

**Pregunta B3.1.** Las curvas de pérdida, ¿muestran que el falsificador está mejorando? ¿Qué gráfico sí lo muestra?

**Respuesta:** (escribe aquí)

**Pregunta B3.2.** El falsificador empieza dibujando ruido. ¿Por qué?

**Respuesta:** (escribe aquí)

**Pregunta B3.3.** ¿En qué se parece y en qué se diferencia una GAN de un modelo de lenguaje como ChatGPT?

**Respuesta:** (escribe aquí)

---
# Entregable

Sube a **AULA-USM** los **dos notebooks estudiante ejecutados**:

1. `Lab12a_Pronostico_estudiante.ipynb`: A1–A4 con respuestas.
2. `Lab12b_MiniGAN_estudiante.ipynb` (este): B1, B2 y B3 con respuestas (el extra en una dimensión no se evalúa).
3. La tabla de abajo, completa.

**¿Qué red uso?** (el árbol de la diapo 23)

> ¿tabla? → **MLP** · ¿imágenes? → **CNN** · ¿datos en orden? → **RNN/LSTM/GRU** · ¿crear datos nuevos? → **GAN**

Completa la columna «Arquitectura» con **MLP, CNN, RNN/LSTM/GRU o GAN** y justifica en una línea:

| Escenario | Arquitectura | Justificación |
|---|---|---|
| **a)** Clasificar la cobertura de suelo (bosque, cultivo, urbano, agua) de imágenes satelitales | | |
| **b)** Pronosticar la demanda eléctrica horaria de una comuna para las próximas 24 horas | | |
| **c)** Generar datos sintéticos de sensores para entrenar un detector de fallas muy poco frecuentes | | |

---
## Extra (opcional) — El juego en una dimensión

*Para quien termine antes. No es parte del entregable.*

> **En palabras simples (diapo 20).** El mismo juego, pero mucho más chico: en vez de dibujar dígitos, el falsificador fabrica **un solo número**. Los números reales forman **dos montes (dos campanas)**, uno alrededor de −2 y otro alrededor de +2. Si el juego sale bien, el falsificador fabrica números en los dos montes. Si el policía aprende demasiado rápido, el falsificador se queda con un solo monte.

Todo viene listo: solo ejecuta las celdas y mira.

In [ ]:
def muestrear_real(n):
    """Dos montes del mismo tamaño: uno en −2 y otro en +2, cada uno una campana de ancho 0,5. Devuelve (n, 1)."""
    elige_izq = np.random.rand(n) < 0.5
    x = np.where(elige_izq, np.random.normal(-2, 0.5, n), np.random.normal(2, 0.5, n))
    return torch.tensor(x, dtype=torch.float32).unsqueeze(1)


fijar_semillas(SEED)
datos_1d = muestrear_real(10_000)
DIM_Z_1D = 8                                # aquí el ruido son 8 números al azar
PASOS_1D = 1000 if RAPIDO else 3000
xs = np.linspace(-5, 5, 400)


def densidad_real(x):
    """La forma exacta de los dos montes (para dibujarla debajo de los histogramas)."""
    campana = lambda centro: np.exp(-0.5 * ((x - centro) / 0.5) ** 2) / (0.5 * np.sqrt(2 * np.pi))
    return 0.5 * campana(-2) + 0.5 * campana(2)


def cobertura_modos(muestras):
    """Fracción de números a menos de 1 del monte −2, del monte +2 y fuera de ambos."""
    m = np.asarray(muestras).ravel()
    izq, der = np.mean(np.abs(m + 2) < 1), np.mean(np.abs(m - 2) < 1)
    return {"monte −2": izq, "monte +2": der, "fuera de ambos": 1 - izq - der}


def dibujar_1d(ax, muestras, titulo):
    """Datos reales en gris claro con borde azul marino; lo fabricado, histograma naranja."""
    ax.fill_between(xs, densidad_real(xs), color=GRIS_CLARO, label="real")
    ax.plot(xs, densidad_real(xs), color=MARINO, lw=1.6)
    ax.hist(np.asarray(muestras).ravel(), bins=60, range=(-5, 5), density=True, color=NARANJA, alpha=0.75, label="fabricado")
    ax.set(xlim=(-5, 5), ylim=(0, 1.0), xlabel="valor", ylabel="densidad", title=titulo)
    ax.legend(loc="upper right", frameon=False, fontsize=9)


def entrenar_gan_1d(pasos=PASOS_1D, lr_g=1e-3, lr_d=1e-3, lote=128, foto_cada=100, semilla=SEED):
    """Los mismos turnos de B1, con un falsificador de 8 -> 32 -> 1 y un policía de 1 -> 32 -> 1.
    Devuelve una «foto» de 5000 números fabricados cada `foto_cada` pasos."""
    fijar_semillas(semilla)
    G = nn.Sequential(nn.Linear(DIM_Z_1D, 32), nn.ReLU(), nn.Linear(32, 1))
    D = nn.Sequential(nn.Linear(1, 32), nn.LeakyReLU(0.2), nn.Linear(32, 1), nn.Sigmoid())
    opt_g = torch.optim.Adam(G.parameters(), lr=lr_g)
    opt_d = torch.optim.Adam(D.parameters(), lr=lr_d)
    bce = nn.BCELoss()
    unos, ceros = torch.ones(lote, 1), torch.zeros(lote, 1)
    fotos = {}
    for paso in range(1, pasos + 1):
        real = datos_1d[torch.randint(0, len(datos_1d), (lote,))]
        falso = G(torch.randn(lote, DIM_Z_1D))
        opt_d.zero_grad()                                       # turno del policía
        (bce(D(real), unos) + bce(D(falso.detach()), ceros)).backward()
        opt_d.step()
        opt_g.zero_grad()                                       # turno del falsificador
        bce(D(falso), unos).backward()
        opt_g.step()
        if paso % foto_cada == 0:
            with torch.no_grad():
                fotos[paso] = G(torch.randn(5000, DIM_Z_1D)).numpy().ravel()
    return fotos


fotos_1d = entrenar_gan_1d()
pasos_1d = sorted(fotos_1d)
print("Cobertura al final:", {k: round(float(v), 2) for k, v in cobertura_modos(fotos_1d[pasos_1d[-1]]).items()},
      "| datos reales:", {k: round(float(v), 2) for k, v in cobertura_modos(datos_1d.numpy()).items()})

Los dos montes apareciendo, un cuadro cada 100 pasos:

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.3), dpi=80)
fig.subplots_adjust(bottom=0.17)


def cuadro_1d(k):
    ax.cla()
    dibujar_1d(ax, fotos_1d[pasos_1d[k]], f"paso {pasos_1d[k]} de {PASOS_1D}")


anim_1d = animation.FuncAnimation(fig, cuadro_1d, frames=len(pasos_1d), interval=300)
plt.close(fig)
HTML(anim_1d.to_jshtml())

**El policía demasiado rápido.** La misma receta, pero el policía aprende 10 veces más rápido (`lr_d=1e-2`) que el falsificador (`lr_g=1e-3`).

In [ ]:
fotos_rapido = entrenar_gan_1d(lr_d=1e-2, lr_g=1e-3)
fotos_rapido_s1 = entrenar_gan_1d(lr_d=1e-2, lr_g=1e-3, semilla=1)     # el mismo experimento, con otra semilla
ultimo = pasos_1d[-1]

fig, ejes = plt.subplots(1, 2, figsize=(11.5, 3.6), sharey=True)
dibujar_1d(ejes[0], fotos_1d[ultimo], f"Los dos a 1e-3: sano (paso {ultimo})")
dibujar_1d(ejes[1], fotos_rapido[ultimo], f"Policía 10 veces más rápido: colapso (paso {ultimo})")
plt.tight_layout()
plt.show()

display(pd.DataFrame({
    "sano (lr_d = 1e-3)": cobertura_modos(fotos_1d[ultimo]),
    "policía rápido (lr_d = 1e-2)": cobertura_modos(fotos_rapido[ultimo]),
    "policía rápido, semilla 1": cobertura_modos(fotos_rapido_s1[ultimo]),
    "datos reales": cobertura_modos(datos_1d.numpy()),
}).T.round(2))

**Pregunta extra.** ¿Cuántos montes aprendió el falsificador cuando el policía aprendía 10 veces más rápido?

**Respuesta:** (escribe aquí)

In [ ]:
print(f"Tiempo total del notebook: {time.time() - T0:.0f} s")

---
## Para saber más (opcional): la fórmula del juego, en palabras

Nada de esto es necesario para el lab. Es lo mismo del Anexo B del deck (diapo 25).

$$\min_G\ \max_D\ \ \mathbb{E}_{x\,\sim\,\text{datos}}\big[\log D(x)\big] \;+\; \mathbb{E}_{z\,\sim\,\text{ruido}}\big[\log\big(1 - D(G(z))\big)\big]$$

| Pieza | En palabras |
|---|---|
| $D(x)$ | Lo que dice el policía sobre un dígito real $x$: de 0 a 1, qué tan seguro está de que es real. |
| $G(z)$ | El dígito falso que dibuja el falsificador a partir del ruido $z$ (los 16 números al azar). |
| $D(G(z))$ | Lo que dice el policía sobre un dígito falso. |
| $\mathbb{E}[\ \cdot\ ]$ | «En promedio», sobre muchos ejemplos (en el código: el promedio de un lote). |
| $\log D(x)$ | Premio del policía por reconocer lo real: vale 0 si dice 1, y se hunde hacia −∞ si dice 0. |
| $\log(1 - D(G(z)))$ | Premio del policía por descubrir lo falso: vale 0 si dice 0 ante un falso. |
| $\max_D$ | El policía quiere que la suma sea **lo más alta posible** (acertar en los dos casos). |
| $\min_G$ | El falsificador quiere lo contrario: que sea **lo más baja posible** (que el policía se equivoque con lo falso). |

En el código no se escribe así: `bce` es ese mismo premio con el signo cambiado (una pérdida, que se baja). Además, en su turno el falsificador usa el truco de pedirle al policía que diga 1 sobre lo falso (`bce(D(falso), unos)`), que da pistas más fuertes al comienzo, cuando el policía todavía lo pilla con facilidad.

**¿Por qué el empate ideal es 1,39 y 0,69?** En el empate, el policía ya no distingue y responde 0,5 a todo. La pérdida `bce` de una respuesta de 0,5 vale $-\ln 0{,}5 = \ln 2 \approx 0{,}69$, sea el dígito real o falso. El falsificador tiene un solo error ($\approx 0{,}69$); el policía suma dos, uno por lo real y otro por lo falso ($2 \ln 2 \approx 1{,}39$).